In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
data = pd.read_excel('Online Retail.xlsx')
data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [3]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
data.shape

(541909, 8)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [6]:
data_clean = data[
    data["Description"].notna()
    & (data["Description"].astype(str).str.strip() != "")
    & (~data["InvoiceNo"].astype(str).str.startswith("C"))
    & (data["Quantity"] > 0)
].copy()

In [7]:
data_clean["Description"] = data_clean["Description"].astype(str).str.strip()
data_clean = data_clean.reset_index(drop=True)

In [8]:
print("SEBELUM PEMBERSIHAN:")
print(f"- Jumlah baris   : {len(data)}")
print(f"- Jumlah invoice : {data['InvoiceNo'].nunique()}\n")

print("SESUDAH PEMBERSIHAN:")
print(f"- Jumlah baris   : {len(data_clean)}")
print(f"- Jumlah invoice : {data_clean['InvoiceNo'].nunique()}")

SEBELUM PEMBERSIHAN:
- Jumlah baris   : 541909
- Jumlah invoice : 25900

SESUDAH PEMBERSIHAN:
- Jumlah baris   : 530693
- Jumlah invoice : 20136


In [9]:
transaksi_invoice = data_clean.groupby("InvoiceNo")["Description"].apply(list)
print("Daftar produk per transaksi (5 invoice pertama):")
print(transaksi_invoice.head(), "\n")


Daftar produk per transaksi (5 invoice pertama):
InvoiceNo
536365    [WHITE HANGING HEART T-LIGHT HOLDER, WHITE MET...
536366    [HAND WARMER UNION JACK, HAND WARMER RED POLKA...
536367    [ASSORTED COLOUR BIRD ORNAMENT, POPPY'S PLAYHO...
536368    [JAM MAKING SET WITH JARS, RED COAT RACK PARIS...
536369                           [BATH BUILDING BLOCK WORD]
Name: Description, dtype: object 



In [10]:
basket = (
    data_clean.groupby(["InvoiceNo", "Description"])["Quantity"]
    .sum()
    .unstack(fill_value=0)
    > 0
)
print(basket.astype(int).head())

Description  *Boombox Ipod Classic  *USB Office Mirror Ball  \
InvoiceNo                                                     
536365                           0                        0   
536366                           0                        0   
536367                           0                        0   
536368                           0                        0   
536369                           0                        0   

Description  10 COLOUR SPACEBOY PEN  12 COLOURED PARTY BALLOONS  \
InvoiceNo                                                         
536365                            0                           0   
536366                            0                           0   
536367                            0                           0   
536368                            0                           0   
536369                            0                           0   

Description  12 DAISY PEGS IN WOOD BOX  12 EGG HOUSE PAINTED WOOD  \
InvoiceNo          

In [ ]:
from mlxtend.frequent_patterns import association_rules, fpgrowth


min_support = [0.01, 0.02, 0.03]

for x in min_support:
    print(f"min_support = {x}")
    frequent_itemsets = fpgrowth(basket, min_support=x, use_colnames=True)
    rules = association_rules(
        frequent_itemsets, metric="confidence", min_threshold=0.5
    )

    print(f"Jumlah frequent itemsets ditemukan : {len(frequent_itemsets)}")
    print(f"Jumlah aturan asosiasi terbentuk   : {len(rules)}")
    display(frequent_itemsets.head(10))
    display(rules[["antecedents", "consequents", "support", "confidence", "lift"]])


min_support = 0.01
Jumlah frequent itemsets ditemukan : 1853
Jumlah aturan asosiasi terbentuk   : 841


,support,itemsets
0,0.112237,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.022795,(KNITTED UNION FLAG HOT WATER BOTTLE)
2,0.021305,(RED WOOLLY HOTTIE WHITE HEART.)
3,0.018872,(SET 7 BABUSHKA NESTING BOXES)
4,0.015246,(WHITE METAL LANTERN)
5,0.014104,(CREAM CUPID HEARTS COAT HANGER)
6,0.023788,(HAND WARMER UNION JACK)
7,0.072259,(ASSORTED COLOUR BIRD ORNAMENT)
8,0.038588,(HOME BUILDING BLOCK WORD)
9,0.031238,(LOVE BUILDING BLOCK WORD)


,antecedents,consequents,support,confidence,lift
0,(POPPY'S PLAYHOUSE KITCHEN),(POPPY'S PLAYHOUSE BEDROOM),0.015544,0.717890,34.173596
1,(POPPY'S PLAYHOUSE BEDROOM),(POPPY'S PLAYHOUSE KITCHEN),0.015544,0.739953,34.173596
2,"(JAM MAKING SET WITH JARS, SET OF 3 CAKE TINS ...",(JAM MAKING SET PRINTED),0.010230,0.542105,9.394003
3,"(SET OF 3 CAKE TINS PANTRY DESIGN, JAM MAKING ...",(JAM MAKING SET WITH JARS),0.010230,0.593660,10.550694
4,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.031784,0.608944,12.511932
...,...,...,...,...,...
836,(SET OF 6 SNACK LOAF BAKING CASES),(SET OF 12 MINI LOAF BAKING CASES),0.011671,0.663842,30.106123
837,(SET OF 6 SNACK LOAF BAKING CASES),(SET OF 6 TEA TIME BAKING CASES),0.010280,0.584746,31.149314
838,(SET OF 6 TEA TIME BAKING CASES),(SET OF 6 SNACK LOAF BAKING CASES),0.010280,0.547619,31.149314
839,(JUMBO BAG VINTAGE DOILY),(JUMBO BAG RED RETROSPOT),0.017680,0.500703,4.819388


min_support = 0.02
Jumlah frequent itemsets ditemukan : 376
Jumlah aturan asosiasi terbentuk   : 61


,support,itemsets
0,0.112237,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.022795,(KNITTED UNION FLAG HOT WATER BOTTLE)
2,0.021305,(RED WOOLLY HOTTIE WHITE HEART.)
3,0.023788,(HAND WARMER UNION JACK)
4,0.072259,(ASSORTED COLOUR BIRD ORNAMENT)
5,0.038588,(HOME BUILDING BLOCK WORD)
6,0.031238,(LOVE BUILDING BLOCK WORD)
7,0.029797,(DOORMAT NEW ENGLAND)
8,0.022298,(FELTCRAFT PRINCESS CHARLOTTE DOLL)
9,0.021653,(POPPY'S PLAYHOUSE KITCHEN)


,antecedents,consequents,support,confidence,lift
0,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.031784,0.608944,12.511932
1,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.031784,0.653061,12.511932
2,(ALARM CLOCK BAKELIKE PINK),(ALARM CLOCK BAKELIKE RED),0.023341,0.600255,11.500231
3,(ALARM CLOCK BAKELIKE PINK),(ALARM CLOCK BAKELIKE GREEN),0.020759,0.533844,10.968864
4,(WOODEN FRAME ANTIQUE WHITE),(WOODEN PICTURE FRAME WHITE FINISH),0.026768,0.555098,10.161318
...,...,...,...,...,...
56,(JUMBO BAG SCANDINAVIAN BLUE PAISLEY),(JUMBO BAG RED RETROSPOT),0.022249,0.676737,6.513757
57,(JUMBO BAG APPLES),(JUMBO BAG RED RETROSPOT),0.027116,0.558282,5.373600
58,(JUMBO BAG PEARS),(JUMBO BAG APPLES),0.020163,0.682353,14.048935
59,(GARDENERS KNEELING PAD KEEP CALM),(GARDENERS KNEELING PAD CUP OF TEA),0.027116,0.598028,15.886413


min_support = 0.03
Jumlah frequent itemsets ditemukan : 136
Jumlah aturan asosiasi terbentuk   : 11


,support,itemsets
0,0.112237,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.072259,(ASSORTED COLOUR BIRD ORNAMENT)
2,0.038588,(HOME BUILDING BLOCK WORD)
3,0.031238,(LOVE BUILDING BLOCK WORD)
4,0.056267,(JAM MAKING SET WITH JARS)
5,0.055920,(POSTAGE)
6,0.052195,(ALARM CLOCK BAKELIKE RED)
7,0.048669,(ALARM CLOCK BAKELIKE GREEN)
8,0.044696,(SPACEBOY LUNCH BOX)
9,0.042163,(RED TOADSTOOL LED NIGHT LIGHT)


,antecedents,consequents,support,confidence,lift
0,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.031784,0.608944,12.511932
1,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.031784,0.653061,12.511932
2,(JUMBO BAG PINK POLKADOT),(JUMBO BAG RED RETROSPOT),0.040971,0.677340,6.519558
3,(JUMBO STORAGE BAG SUKI),(JUMBO BAG RED RETROSPOT),0.035956,0.611486,5.885704
4,(JUMBO SHOPPER VINTAGE RED PAISLEY),(JUMBO BAG RED RETROSPOT),0.033770,0.578723,5.570351
5,(LUNCH BAG PINK POLKADOT),(LUNCH BAG RED RETROSPOT),0.030095,0.555963,7.157850
6,(LUNCH BAG BLACK SKULL.),(LUNCH BAG RED RETROSPOT),0.031834,0.503535,6.482852
7,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.038141,0.756650,14.292598
8,(ROSES REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.038141,0.720450,14.292598
9,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.031436,0.826371,16.393893


In [12]:
frequent_itemsets = fpgrowth(basket, min_support=0.03, use_colnames=True)
rules = association_rules(
    frequent_itemsets, metric="confidence", min_threshold=0.5
)

print(f"Jumlah frequent itemsets ditemukan : {len(frequent_itemsets)}")
print(f"Jumlah aturan asosiasi terbentuk   : {len(rules)}")
display(frequent_itemsets.head(10))
display(rules[["antecedents", "consequents", "support", "confidence", "lift"]])

Jumlah frequent itemsets ditemukan : 136
Jumlah aturan asosiasi terbentuk   : 11


,support,itemsets
0,0.112237,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.072259,(ASSORTED COLOUR BIRD ORNAMENT)
2,0.038588,(HOME BUILDING BLOCK WORD)
3,0.031238,(LOVE BUILDING BLOCK WORD)
4,0.056267,(JAM MAKING SET WITH JARS)
5,0.055920,(POSTAGE)
6,0.052195,(ALARM CLOCK BAKELIKE RED)
7,0.048669,(ALARM CLOCK BAKELIKE GREEN)
8,0.044696,(SPACEBOY LUNCH BOX)
9,0.042163,(RED TOADSTOOL LED NIGHT LIGHT)


,antecedents,consequents,support,confidence,lift
0,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.031784,0.608944,12.511932
1,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.031784,0.653061,12.511932
2,(JUMBO BAG PINK POLKADOT),(JUMBO BAG RED RETROSPOT),0.040971,0.677340,6.519558
3,(JUMBO STORAGE BAG SUKI),(JUMBO BAG RED RETROSPOT),0.035956,0.611486,5.885704
4,(JUMBO SHOPPER VINTAGE RED PAISLEY),(JUMBO BAG RED RETROSPOT),0.033770,0.578723,5.570351
5,(LUNCH BAG PINK POLKADOT),(LUNCH BAG RED RETROSPOT),0.030095,0.555963,7.157850
6,(LUNCH BAG BLACK SKULL.),(LUNCH BAG RED RETROSPOT),0.031834,0.503535,6.482852
7,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.038141,0.756650,14.292598
8,(ROSES REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.038141,0.720450,14.292598
9,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.031436,0.826371,16.393893


min support = 0.03 dipilih agar list tentang barang yang saling berkaitan gampang dibaca dan ringkas juga untuk hanya mengambil hubungan barang yang sering dibeli agar hasilnya signifikan untuk toko retail

In [ ]:
kolom_evaluasi = ["antecedents", "consequents", "support", "confidence", "lift"]
print(f"Total aturan asosiasi yang dihasilkan: {len(rules)}")
display(rules[kolom_evaluasi])

rules_unique = rules.copy()
rules_unique["item_pair"] = rules_unique.apply(
    lambda row: frozenset(row["antecedents"] | row["consequents"]), axis=1
)

top_3_rules = (
    rules_unique.sort_values(
        by=["lift", "confidence"], ascending=[False, False]
    )
    .drop_duplicates(subset="item_pair", keep="first")
    .drop(columns="item_pair")[kolom_evaluasi]
    .head(3)
    .reset_index(drop=True)
)
print("\n3 Aturan Terbaik Berdasarkan Nilai Lift (Tanpa Redundansi):")
display(top_3_rules)

Total aturan asosiasi yang dihasilkan: 11


,antecedents,consequents,support,confidence,lift
0,(ALARM CLOCK BAKELIKE RED),(ALARM CLOCK BAKELIKE GREEN),0.031784,0.608944,12.511932
1,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.031784,0.653061,12.511932
2,(JUMBO BAG PINK POLKADOT),(JUMBO BAG RED RETROSPOT),0.040971,0.677340,6.519558
3,(JUMBO STORAGE BAG SUKI),(JUMBO BAG RED RETROSPOT),0.035956,0.611486,5.885704
4,(JUMBO SHOPPER VINTAGE RED PAISLEY),(JUMBO BAG RED RETROSPOT),0.033770,0.578723,5.570351
5,(LUNCH BAG PINK POLKADOT),(LUNCH BAG RED RETROSPOT),0.030095,0.555963,7.157850
6,(LUNCH BAG BLACK SKULL.),(LUNCH BAG RED RETROSPOT),0.031834,0.503535,6.482852
7,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.038141,0.756650,14.292598
8,(ROSES REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.038141,0.720450,14.292598
9,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.031436,0.826371,16.393893



3 Aturan Terbaik Berdasarkan Nilai Lift (Tanpa Redundansi):


,antecedents,consequents,support,confidence,lift
0,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.031436,0.826371,16.393893
1,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.038141,0.756650,14.292598
2,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.031784,0.653061,12.511932


In [14]:
for i, row in top_3_rules.iterrows():
  ant = ", ".join(list(row["antecedents"]))
  con = ", ".join(list(row["consequents"]))
  sup = row["support"]
  conf = row["confidence"]
  lift = row["lift"]

  print(f"Aturan #{i+1}: Jika membeli [{ant}] -> Maka membeli [{con}]")
  print(f"  - Support    : {sup:.4f} ({sup*100:.2f}% dari seluruh transaksi)")
  print(
      f"  - Confidence : {conf:.4f} ({conf*100:.2f}% pembeli [{ant}] juga"
      f" membeli [{con}])"
  )
  print(
      f"  - Lift       : {lift:.2f} (Pembelian bersama {lift:.2f}x lebih tinggi"
      " dibanding jika dibeli secara acak/independen)\n"
  )

Aturan #1: Jika membeli [PINK REGENCY TEACUP AND SAUCER] -> Maka membeli [GREEN REGENCY TEACUP AND SAUCER]
  - Support    : 0.0314 (3.14% dari seluruh transaksi)
  - Confidence : 0.8264 (82.64% pembeli [PINK REGENCY TEACUP AND SAUCER] juga membeli [GREEN REGENCY TEACUP AND SAUCER])
  - Lift       : 16.39 (Pembelian bersama 16.39x lebih tinggi dibanding jika dibeli secara acak/independen)

Aturan #2: Jika membeli [GREEN REGENCY TEACUP AND SAUCER] -> Maka membeli [ROSES REGENCY TEACUP AND SAUCER]
  - Support    : 0.0381 (3.81% dari seluruh transaksi)
  - Confidence : 0.7567 (75.67% pembeli [GREEN REGENCY TEACUP AND SAUCER] juga membeli [ROSES REGENCY TEACUP AND SAUCER])
  - Lift       : 14.29 (Pembelian bersama 14.29x lebih tinggi dibanding jika dibeli secara acak/independen)

Aturan #3: Jika membeli [ALARM CLOCK BAKELIKE GREEN] -> Maka membeli [ALARM CLOCK BAKELIKE RED]
  - Support    : 0.0318 (3.18% dari seluruh transaksi)
  - Confidence : 0.6531 (65.31% pembeli [ALARM CLOCK BAKELIKE G

Dari 3 aturan terbaik dapat dibuat bundling barang karena jika seseorang membeli barang pertama maka kemungkinan membeli barang ke 2 sangat besar jadi jika diberi bundling dengan diskon maka kemungkinan terjualnya banyak. Jika dilihat sebenarnya dari 3 aturan itu sama sama membeli 2 barang yang sama namun dengan varian warna berbeda sehingga dapat disimpulkan bahwa kebanyakan pengunjung cenderung membeli barang yang sama namun dengan varian berbeda tiap berkunjung sehingga barang yang satu kategori disarankan dipasang berjajar